### Notebook to make final adjustments to subglacial discharge CESM-WACCM processing

This notebook makes the following adjustments to the CESM-WACCM runoff files:
1. Making time axis consistent with TF files
2. Splitting into annual files
3. Fixing attributes (which look like they were stripped out at the bias correction stage)
4. Setting final file naming convention
5. Adding a 2300 file that is the same as 2299

### Setup

In [1]:
# To run this notebook, uncomment one of the following setup blocks (historical, projection, or extension)

# historical
Qdir = '~/Documents/CESM2-WACCM/historical/runoff/'
Qfile = 'Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc'
Qroot = 'historical'
# Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/historical/PROCESSEDmon/runoff/SDBN1'
# Qfile = 'Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc'
# Qroot = 'historical'
# projection
#Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/runoff/SDBN1'
#Qfile = 'Q_CESM2-WACCM_ssp585_SDBN1_2015_2100_biascorrected.nc'
#Qroot = 'ssp585'
# extension
# Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/runoff/SDBN1'
# Qfile = 'Q_CESM2-WACCM_ssp585_SDBN1_2101_2299_biascorrected.nc'
# Qroot = 'ssp585'

# CESM-WACCM output file for year 2299
Q2299 = '~/Documents/CESM2-WACCM/ssp585/runoff/extension/Q_CESM2-WACCM_ssp585_SDBN1_2299.nc'


### Imports

In [2]:
import xarray as xr
import cftime
import os
import numpy as np

In [ ]:
# open file
ds_Q = xr.open_dataset(Qdir+Qfile,decode_times=False)

# make timestamps cf compliant by converting from months since Jan 1850 to cftime.DatetimeNoLeap objects
# fixes timestamp at middle of month
months_since_jan1850 = ds_Q['time'].values
days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
dates = []
for m in months_since_jan1850:
    m = int(m)
    year  = 1850 + m // 12
    month = m % 12 + 1
    day = np.ceil(days_in_month[month-1] / 2)
    dates.append(cftime.DatetimeNoLeap(year, month, day))

ds_Q = ds_Q.assign_coords(time=dates)

# fix attributes that seem to have been stripped out when doing the bias correction
ds_Q['Q'].attrs['long_name'] = 'basin subglacial discharge, monthly mean'
ds_Q['Q'].attrs['units'] = 'm3 s-1'
ds_Q['Q'].attrs['grid_mapping'] = 'crs'
ds_Q['time'].attrs['standard_name'] = 'time'
ds_Q.attrs['Conventions'] = 'CF-1.8'

# write out files in annual chunks
startyear = ds_Q['time'].values[0].year
endyear = ds_Q['time'].values[-1].year
#endyear = 2018
for year in range(startyear,endyear+1):
    ds_Q_year = ds_Q.sel(time=slice(cftime.DatetimeNoLeap(year,1,1),cftime.DatetimeNoLeap(year,12,31)))
    outfile = Qdir+'Q_CESM2-WACCM_'+Qroot+'_SDBN1_'+str(year)+'.nc'
    # remove file if it exists
    if os.path.exists(outfile):
        os.remove(outfile)
    print('writing '+outfile)
    ds_Q_year.to_netcdf(outfile)

writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1850.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1851.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1852.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1853.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1854.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1855.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1856.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1857.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1858.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1859.nc
writing ~/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1860.nc
writing ~/Documents/C

In [ ]:
# add a 2300 file that is the same as 2299 but with updated timestamp
ds_2299 = xr.open_dataset(Q2299)
ds_2300 = ds_2299.copy()

# replace year of time coordinate to 2300
new_times = []
for t in ds_2299['time'].values:
    new_time = cftime.DatetimeNoLeap(2300, t.month, t.day)
    new_times.append(new_time)
ds_2300 = ds_2300.assign_coords(time=new_times)

# save out
outfile = Q2299.replace('2299','2300')
if os.path.exists(outfile):
    os.remove(outfile)
ds_2300.to_netcdf(outfile)
